# Student 3 — Image Analyst: Crime/Incident Scene Object Detection

**Assignment:** AI for Engineers — Multimodal Crime / Incident Report Analyzer  
**Role:** Student 3 — Image Analyst  
**Pipeline:** Load images → OpenCV preprocessing → Roboflow YOLOv8 fire detection → pytesseract OCR → Scene classification → Export CSV

### Required Output Columns
`Image_ID | Scene_Type | Objects_Detected | Bounding_Boxes | Confidence`

---

## Dataset
**Roboflow Fire Detection** — pre-labeled fire and smoke images, YOLOv8 format  
- Project: `my-space-3zzwr/fire-smoke-detection-odvk6` (version 1)  
- 1000+ fire and smoke images, fine-tuned YOLOv8 model  
- Classes detected: `fire`, `smoke`, `other`

In [ ]:
# ============================================================
# CELL 1 — Install all dependencies
# ============================================================
!pip install ultralytics opencv-python pytesseract roboflow -q
print('All packages installed.')

In [ ]:
# ============================================================
# CELL 2 — Download Roboflow Fire Detection Dataset
# ============================================================
from roboflow import Roboflow
import os

RF_API_KEY = "JrpwBu99OhGyhR1Gcqy0"

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace('my-space-3zzwr').project('fire-smoke-detection-odvk6')
version = project.version(1)
dataset = version.download('yolov8')

IMAGE_DIR = os.path.join(dataset.location, 'test', 'images')
print(f'Dataset downloaded. Test images at: {IMAGE_DIR}')
print(f'Sample images: {os.listdir(IMAGE_DIR)[:5]}')

In [ ]:
# ============================================================
# CELL 2b — List images to process (cap at 20 for speed)
# ============================================================
import os

image_files = [f for f in os.listdir(IMAGE_DIR)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
image_files = sorted(image_files)[:20]   # process first 20 test images
print(f'Processing {len(image_files)} fire/smoke images')

In [ ]:
# ============================================================
# CELL 3 — OpenCV image preprocessing
# Resize, denoise, enhance contrast for better detection
# ============================================================
import cv2
import numpy as np
import os

PROCESSED_DIR = 'processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

def preprocess_image(img_path, target_size=(640, 640)):
    """Resize, denoise, normalize for YOLOv8 inference."""
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f'Cannot read: {img_path}')
    # Resize to YOLOv8 standard input
    img = cv2.resize(img, target_size)
    # Denoise
    img = cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)
    # CLAHE contrast enhancement on L channel
    lab   = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab   = cv2.merge([clahe.apply(l), a, b])
    img   = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    return img

processed_paths = {}
for img_file in image_files:
    src = os.path.join(IMAGE_DIR, img_file)
    dst = os.path.join(PROCESSED_DIR, img_file)
    img = preprocess_image(src)
    cv2.imwrite(dst, img)
    processed_paths[img_file] = dst

print(f'Preprocessed {len(processed_paths)} images.')

In [ ]:
# ============================================================
# CELL 4 — Roboflow Fire-Smoke Detection (hosted model)
# Uses the fine-tuned fire/smoke YOLOv8 model from Roboflow.
# Classes: fire, smoke, other  (NOT the generic COCO 80-class model)
# ============================================================

# Use the Roboflow hosted inference model — detects fire and smoke
model = version.model   # fire-smoke-detection-odvk6 v1

SCENE_MAP = [
    (['fire', 'smoke'],                              'Fire Scene'),
    (['gun', 'knife', 'weapon', 'pistol'],           'Crime / Assault Scene'),
    (['car', 'truck', 'bus', 'motorcycle',
      'bicycle', 'stop sign', 'traffic light'],      'Road Incident Scene'),
    (['person'],                                     'Public Disturbance Scene'),
]

def classify_scene(objects):
    obj_lower = [o.lower() for o in objects]
    for keywords, label in SCENE_MAP:
        if any(k in obj_lower for k in keywords):
            return label
    return 'General Incident Scene'

detection_results = {}
for img_file, proc_path in processed_paths.items():
    try:
        pred = model.predict(proc_path, confidence=30, overlap=30).json()
        detected, confs, bboxes = [], [], []
        for p in pred.get('predictions', []):
            detected.append(p['class'])
            confs.append(round(p['confidence'], 2))
            x, y, w, h = p['x'], p['y'], p['width'], p['height']
            bboxes.append(f'[{int(x-w/2)},{int(y-h/2)},{int(x+w/2)},{int(y+h/2)}]')
        detection_results[img_file] = {
            'detected': detected, 'confs': confs, 'bboxes': bboxes
        }
        print(f'{img_file[:40]}: {set(detected)} | avg_conf={round(sum(confs)/max(len(confs),1),2)}')
    except Exception as e:
        print(f'ERROR {img_file}: {e}')
        detection_results[img_file] = {'detected': [], 'confs': [], 'bboxes': []}

In [ ]:
# ============================================================
# CELL 5 — pytesseract OCR: extract text from images
# ============================================================
import pytesseract
import cv2, re

def extract_text_from_image(img_path):
    img  = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
    text = pytesseract.image_to_string(thresh, config='--psm 11').strip()
    text = re.sub(r'[^\x20-\x7E\n]', '', text)
    lines = [l.strip() for l in text.splitlines() if len(l.strip()) >= 2]
    return ' | '.join(lines) if lines else 'none'

ocr_results = {}
for img_file, proc_path in processed_paths.items():
    try:
        ocr_results[img_file] = extract_text_from_image(proc_path)
    except Exception as e:
        ocr_results[img_file] = 'none'
print('OCR complete.')

In [ ]:
# ============================================================
# CELL 6 — Build structured DataFrame
# ============================================================
import pandas as pd

rows = []
for i, img_file in enumerate(image_files):
    det      = detection_results.get(img_file, {})
    detected = det.get('detected', [])
    confs    = det.get('confs', [])
    bboxes   = det.get('bboxes', [])

    img_id     = 'IMG_' + os.path.splitext(img_file)[0][:20].upper().replace('-','_').replace('.','_')
    scene_type = classify_scene(detected)
    obj_str    = ', '.join(dict.fromkeys(detected)) if detected else 'none'
    bbox_str   = '; '.join(bboxes) if bboxes else 'none'
    avg_conf   = round(sum(confs) / max(len(confs), 1), 2)

    rows.append({
        'Image_ID':         img_id,
        'Scene_Type':       scene_type,
        'Objects_Detected': obj_str,
        'Bounding_Boxes':   bbox_str,
        'Confidence':       avg_conf,
    })

image_df = pd.DataFrame(rows)
print(image_df.to_string(index=False))

In [ ]:
# ============================================================
# CELL 7 — Validate and save image_output.csv
# ============================================================

REQUIRED_COLUMNS = ['Image_ID', 'Scene_Type', 'Objects_Detected', 'Bounding_Boxes', 'Confidence']

image_df = image_df[REQUIRED_COLUMNS]

assert list(image_df.columns) == REQUIRED_COLUMNS
assert len(image_df) > 0

image_df.to_csv('image_output.csv', index=False)

print(f'Saved image_output.csv')
print(f'  Rows: {len(image_df)} | Fire Scene: {(image_df["Scene_Type"]=="Fire Scene").sum()}')
print(image_df.to_string(index=False))